# 5.3 SpMV：`acl.op.execute_v2`

## 小节概述

目标：先用 ATC 编译 `SparseTensorDenseMatMul` 单算子 OM，再通过 `acl.op.execute_v2` 提交 COO SpMV，并用 COO 累加结果校验。

<img src="images/spmv_gemm_data_contract.svg" width="760" style="display:block; margin-left:0;" />

<table style="text-align:left; margin-left:0;">
<tr><th>输入</th><th>合同</th><th>失败条件</th></tr>
<tr><td>indices</td><td><code>[NNZ,2]</code>，int64，依次为 row/col</td><td>越界、shape 错误；本实验还禁止重复坐标</td></tr>
<tr><td>values</td><td><code>[NNZ]</code>，float32</td><td>长度与 indices 不一致</td></tr>
<tr><td>dense_shape</td><td><code>[M,N]</code>，int64</td><td>与 x/output 不一致</td></tr>
<tr><td>x2 / output</td><td><code>[N,1]</code> / <code>[M,1]</code>，float32</td><td>维度或 dtype 不匹配</td></tr>
</table>

> **后端边界：**本课程在 910B3/CANN 9.0 上把该算子标记为 AI CPU/tf_kernel 能力路径，`aicore_accelerated=false`。不能把算子在 NPU 系统中执行写成 AI Core 加速。


In [ ]:
import json
import subprocess
import time
from pathlib import Path

import acl
import numpy as np

SOC_VERSION = acl.get_soc_name()

M, N, NNZ = 128, 128, 256
DEVICE_ID = 0
REPEAT = 10
MODEL_DIR = (Path("work") / "01.03_spmv" / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"M={M}, N={N}, NNZ={NNZ}, values/x/y=float32, device={DEVICE_ID}, repeat={REPEAT}, soc={SOC_VERSION}")


## 生成并编译单算子描述

ATC 根据下面的输入、输出和属性描述生成单算子 OM；输入顺序必须与 `SparseTensorDenseMatMul` 的算子合同一致。


In [ ]:
MODEL_DIR = (Path("work") / "01.03_spmv" / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

spmv_singleop = [{
    "op": "SparseTensorDenseMatMul",
    "input_desc": [
        {"format": "ND", "shape": [NNZ, 2], "type": "int64"},  # indices
        {"format": "ND", "shape": [NNZ], "type": "float"},     # values
        {"format": "ND", "shape": [2], "type": "int64"},       # dense_shape
        {"format": "ND", "shape": [N, 1], "type": "float"},    # x2
    ],
    "output_desc": [{"format": "ND", "shape": [M, 1], "type": "float"}],
    "attr": [
        {"name": "adjoint_a", "type": "bool", "value": False},
        {"name": "adjoint_b", "type": "bool", "value": False},
    ],
}]

json_path = MODEL_DIR / "spmv_singleop.json"
json_path.write_text(json.dumps(spmv_singleop, indent=2, ensure_ascii=False), encoding="utf-8")
print("已写入", json_path)


In [ ]:
cmd = [
    "atc", f"--singleop={json_path}", f"--output={MODEL_DIR}",
    f"--soc_version={SOC_VERSION}",
]
print("$", *cmd)
subprocess.run(cmd, check=True)


## 直接调用 SpMV 并校验

先验证 COO shape、坐标范围和唯一性，再依次创建 TensorDesc/DataBuffer，调用 `acl.op.execute_v2`。Event 只统计同一 Stream 上的重复算子时间；D2H 后按 COO 坐标计算 NumPy reference。SpMV 与 GEMM 的数据类型、Kernel 类别和工作量不同，本节不做二者性能横向比较。


In [ ]:
ACL_FLOAT = 0
ACL_INT64 = 9
ACL_FORMAT_ND = 2
MEMCPY_H2D = 1
MEMCPY_D2H = 2
MALLOC_FLAG = 2


def check(ret, call):
    if ret != 0:
        raise RuntimeError(f"{call} 失败，ret={ret}")


linear = np.arange(NNZ, dtype=np.int64) * (M * N // NNZ)
indices = np.stack([linear // N, linear % N], axis=1)
values = ((np.arange(NNZ) * 7 + 1) % 13 + 1).astype(np.float32)
dense_shape = np.array([M, N], dtype=np.int64)
x2 = ((np.arange(N) * 3 + 1) % 11).astype(np.float32).reshape(N, 1)

assert indices.shape == (NNZ, 2) and values.shape == (NNZ,)
assert dense_shape.tolist() == [M, N] and x2.shape == (N, 1)
assert np.all((0 <= indices[:, 0]) & (indices[:, 0] < M))
assert np.all((0 <= indices[:, 1]) & (indices[:, 1] < N))
assert np.unique(indices, axis=0).shape[0] == NNZ, "本实验要求 COO 坐标唯一"

context = stream = start_event = end_event = None
acl_inited = device_set = False
ptrs = []
tensor_objects = []
attr = None


def input_tensor(array, dtype):
    ptr, ret = acl.rt.malloc(array.nbytes, MALLOC_FLAG)
    check(ret, "acl.rt.malloc")
    ptrs.append(ptr)
    check(acl.rt.memcpy(
        ptr, array.nbytes, acl.util.numpy_to_ptr(array), array.nbytes, MEMCPY_H2D
    ), "acl.rt.memcpy H2D")
    desc = acl.create_tensor_desc(dtype, list(array.shape), ACL_FORMAT_ND)
    buffer = acl.create_data_buffer(ptr, array.nbytes)
    if desc is None or buffer is None:
        raise RuntimeError("创建输入 TensorDesc/DataBuffer 失败")
    tensor_objects.append((desc, buffer))
    return desc, buffer


try:
    check(acl.init(), "acl.init")
    acl_inited = True
    check(acl.op.set_model_dir(str(MODEL_DIR)), "acl.op.set_model_dir")
    check(acl.rt.set_device(DEVICE_ID), "acl.rt.set_device")
    device_set = True
    context, ret = acl.rt.create_context(DEVICE_ID)
    check(ret, "acl.rt.create_context")
    stream, ret = acl.rt.create_stream()
    check(ret, "acl.rt.create_stream")
    start_event, ret = acl.rt.create_event()
    check(ret, "acl.rt.create_event start")
    end_event, ret = acl.rt.create_event()
    check(ret, "acl.rt.create_event end")
    current_device, ret = acl.rt.get_device()
    check(ret, "acl.rt.get_device")

    idx_desc, idx_buffer = input_tensor(indices, ACL_INT64)
    val_desc, val_buffer = input_tensor(values, ACL_FLOAT)
    shape_desc, shape_buffer = input_tensor(dense_shape, ACL_INT64)
    x_desc, x_buffer = input_tensor(x2, ACL_FLOAT)

    out_host = np.empty((M, 1), dtype=np.float32)
    out_ptr, ret = acl.rt.malloc(out_host.nbytes, MALLOC_FLAG)
    check(ret, "acl.rt.malloc output")
    ptrs.append(out_ptr)
    out_desc = acl.create_tensor_desc(ACL_FLOAT, [M, 1], ACL_FORMAT_ND)
    out_buffer = acl.create_data_buffer(out_ptr, out_host.nbytes)
    if out_desc is None or out_buffer is None:
        raise RuntimeError("创建输出 TensorDesc/DataBuffer 失败")
    tensor_objects.append((out_desc, out_buffer))

    attr = acl.op.create_attr()
    if attr is None:
        raise RuntimeError("acl.op.create_attr 失败")
    check(acl.op.set_attr_bool(attr, "adjoint_a", False), "set adjoint_a")
    check(acl.op.set_attr_bool(attr, "adjoint_b", False), "set adjoint_b")

    input_descs = [idx_desc, val_desc, shape_desc, x_desc]
    input_buffers = [idx_buffer, val_buffer, shape_buffer, x_buffer]
    check(acl.rt.record_event(start_event, stream), "acl.rt.record_event start")
    for _ in range(REPEAT):
        check(acl.op.execute_v2(
            "SparseTensorDenseMatMul",
            input_descs, input_buffers,
            [out_desc], [out_buffer], attr, stream,
        ), "acl.op.execute_v2")
    check(acl.rt.record_event(end_event, stream), "acl.rt.record_event end")
    check(acl.rt.synchronize_stream(stream), "acl.rt.synchronize_stream")
    elapsed_ms, ret = acl.rt.event_elapsed_time(start_event, end_event)
    check(ret, "acl.rt.event_elapsed_time")
    check(acl.rt.memcpy(
        acl.util.numpy_to_ptr(out_host), out_host.nbytes,
        out_ptr, out_host.nbytes, MEMCPY_D2H,
    ), "acl.rt.memcpy D2H")

    reference_start = time.perf_counter()
    golden = np.zeros(M, dtype=np.float32)
    for (row, col), value in zip(indices, values):
        golden[row] += value * x2[col, 0]
    reference_ms = (time.perf_counter() - reference_start) * 1000.0
    max_abs_error = float(np.max(np.abs(out_host[:, 0] - golden)))
    np.testing.assert_allclose(out_host[:, 0], golden, atol=1e-3, rtol=0)

    spmv_result = {
        "status": "PASS",
        "actual_backend": f"AICPU/tf_kernel capability path on {SOC_VERSION}/CANN 9.0",
        "aicore_accelerated": False,
        "device_id": int(current_device),
        "shape": [M, N],
        "nnz": NNZ,
        "repeat": REPEAT,
        "device_mean_ms": elapsed_ms / REPEAT,
        "reference_ms": reference_ms,
        "max_abs_error": max_abs_error,
        "fallback": 0,
    }
    print("SPMV_REPORT=" + json.dumps(spmv_result, ensure_ascii=False))
finally:
    for event in (end_event, start_event):
        if event is not None:
            acl.rt.destroy_event(event)
    if attr is not None:
        acl.op.destroy_attr(attr)
    for desc, buffer in reversed(tensor_objects):
        acl.destroy_data_buffer(buffer)
        acl.destroy_tensor_desc(desc)
    for ptr in reversed(ptrs):
        acl.rt.free(ptr)
    if stream is not None:
        acl.rt.destroy_stream(stream)
    if context is not None:
        acl.rt.destroy_context(context)
    if device_set:
        acl.rt.reset_device(DEVICE_ID)
    if acl_inited:
        acl.finalize()


## 课后实践

把 `M、N、NNZ` 改为 `64、128、100`，保持 COO 坐标唯一且不越界，重新运行全部 Cell。记录后端分类、Device ID、平均设备时间和最大误差，并说明为什么 reference 可以直接按 `(row, col, value)` 累加。

下面的 Cell 只准备并检查独立输入；完成后仍须通过真实 `acl.op.execute_v2` 路径得到 NPU 输出。


In [ ]:
PRACTICE_M, PRACTICE_N, PRACTICE_NNZ = 64, 128, 100
practice_linear = np.arange(PRACTICE_NNZ, dtype=np.int64) * (
    PRACTICE_M * PRACTICE_N // PRACTICE_NNZ
)
practice_indices = np.stack(
    [practice_linear // PRACTICE_N, practice_linear % PRACTICE_N], axis=1
)
assert np.unique(practice_indices, axis=0).shape[0] == PRACTICE_NNZ
assert np.all(practice_indices[:, 0] < PRACTICE_M)
assert np.all(practice_indices[:, 1] < PRACTICE_N)
print("独立 COO 输入已准备：", practice_indices.shape)


In [ ]:
# 完成练习后按需执行；Notebook 不会自动展开答案。
!cat answer/05.03_spmv_answer.md
